# 🧬 Biomedical Knowledge Graph RAG + Q&A
This notebook demonstrates how to build a Retrieval-Augmented Generation (RAG) system using a Biomedical Knowledge Graph for enhanced Q&A.

In [ ]:
!pip install gradio networkx rdflib tiktoken transformers langchain faiss-cpu scikit-learn sentence-transformers huggingface_hub

In [ ]:

import os
import gradio as gr
import networkx as nx
from rdflib import Graph, URIRef, Literal, RDF, Namespace
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.chains import RetrievalQA
from langchain.llms import HuggingFaceHub
from langchain.text_splitter import CharacterTextSplitter
from langchain.docstore.document import Document


In [ ]:

# Sample biomedical triples (subject, predicate, object)
triples = [
    ("COVID-19", "has_symptom", "Fever"),
    ("COVID-19", "has_symptom", "Cough"),
    ("Remdesivir", "treats", "COVID-19"),
    ("Diabetes", "has_symptom", "Fatigue"),
    ("Insulin", "treats", "Diabetes"),
]

# Build RDF Graph
g = Graph()
BIO = Namespace("http://bio.org/")
for s, p, o in triples:
    g.add((BIO[s.replace(" ", "_")], BIO[p], Literal(o)))

# Serialize RDF to Turtle
g.serialize(destination="/mnt/data/biomedical_kg.ttl", format="turtle")
print(g.serialize(format='turtle'))


In [ ]:

docs = [Document(page_content=f"{s} {p} {o}") for s, p, o in triples]
splitter = CharacterTextSplitter(chunk_size=100, chunk_overlap=10)
split_docs = splitter.split_documents(docs)

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(split_docs, embeddings)

llm = HuggingFaceHub(repo_id="google/flan-t5-large", model_kwargs={"temperature": 0.5, "max_length": 256})
qa_chain = RetrievalQA.from_chain_type(llm=llm, retriever=vectorstore.as_retriever())


In [ ]:

def query_kg(question):
    return qa_chain.run(question)

gr.Interface(
    fn=query_kg,
    inputs=gr.Textbox(label="Ask a biomedical question"),
    outputs=gr.Textbox(label="Answer"),
    title="Biomedical Knowledge Graph RAG Q&A"
).launch()
